# Pipeline de Transformação: devolucoes

In [ ]:
import sys
from pathlib import Path

# Adiciona o diretório raiz do projeto ao sys.path para importações locais
sys.path.append(str(Path.cwd().parent.parent))

from src.modules.spark_session import get_spark_session, close_spark_session
import src.modules.transform_utils as transform

In [ ]:
# Inicializa a SparkSession conectada ao cluster do container
spark = get_spark_session("TransformDevolucoes")

# Leitura dos dados da camada Bronze

In [ ]:
# Caminho da tabela Bronze no MinIO
bronze_path = "s3a://bronze/devolucoes"

# Lê os dados da Bronze
df_devolucoes = spark.read.parquet(bronze_path)

print(f"Total de registros na camada Bronze: {df_devolucoes.count()}")
df_devolucoes.printSchema()
df_devolucoes.limit(5).toPandas()

# Aplica TRIM nas colunas de texto

In [ ]:
text_cols = ["motivo_devolucao", "status_devolucao"]
df_devolucoes = transform.trim_columns(df_devolucoes, text_cols)

df_devolucoes.select("devolucao_id", "motivo_devolucao", "status_devolucao").limit(5).toPandas()

# Cria colunas de ano e mês

In [ ]:
df_devolucoes = transform.extract_date_parts(df_devolucoes, "data_devolucao")

df_devolucoes.select("devolucao_id", "data_devolucao", "ano", "mes").limit(5).toPandas()

In [ ]:
value_cols = ["valor_devolvido"]
df_devolucoes = transform.round_values(df_devolucoes, value_cols, decimals=2)

df_devolucoes.select("devolucao_id", "valor_devolvido").limit(5).toPandas()

In [ ]:
# Finaliza a sessão do Spark
close_spark_session(spark)